# How Skuld's true-positive rate was measured

A detector can always recover more signals by lowering its decision threshold, but it will also create more false detections. This notebook explains how the threshold and true-positive rate (TPR) were measured without using the validation spectra to tune the answer.

The saved results come from the standard paired observing-window campaign. Loading the summary takes seconds; rerunning all 480 adaptive evidence calculations is intentionally left to `examples/model_set_study.py`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import binomtest

plt.style.use('seaborn-v0_8-whitegrid')
data_dir = next(
    path for path in (Path('notebooks/data'), Path('data')) if path.is_dir()
)
summary = json.loads((data_dir / 'standard_calibration_summary.json').read_text())
summary['campaign']

## 1. Where the test spectra come from

The campaign crosses several controlled coordinates:

- three stellar regimes: dwarf, subgiant, and low-luminosity red giant;
- 27.4-day and 90-day observing baselines;
- two white-noise levels per regime;
- noise-only and granulation-only negative cases;
- oscillation amplitudes of $0.3$, $0.5$, and $1.0$ times the AsteroScale prediction; and
- continuous and TESS-like gapped views of the same stochastic realization.

For every cell, AsteroScale-like joint samples supply correlated $\nu_{\max}$, $\Delta\nu$, envelope, and granulation parameters. The injection uses the median star to generate a resolved Lorentzian mode comb, whereas the detector fits a smooth Gaussian power envelope. This deliberate mismatch makes the test less circular.

## 2. Tuning and validation are separated

Independent stochastic realizations of every grid cell are split in half **before inference**:

- the tuning half chooses the probability threshold while requiring a false-positive rate no larger than 5%;
- the validation half is opened only after that threshold is frozen.

This is analogous to adjusting an instrument on calibration data before observing the science target. The tuning population selected

$$P(\mathrm{oscillation}\mid D) \geq 0.45$$

as the detection rule.

In [ ]:
campaign = summary['campaign']
print(f"Frozen threshold: {campaign['selected_threshold']:.2f}")
print(f"Maximum tuning false-positive rate: {campaign['maximum_tuning_false_positive_rate']:.0%}")

## 3. True-positive rate is conditional on signal strength

The true-positive rate is

$$
\mathrm{TPR}=\frac{\mathrm{true\ detections}}{\mathrm{injected\ oscillators}}.
$$

A single overall value mixes easy full-amplitude detections with deliberately difficult suppressed signals. The plot therefore separates the three injected amplitudes. Error bars are 95% Wilson binomial intervals; there are only 24 validation oscillators in each bar, so the uncertainty is still substantial.

In [ ]:
def wilson_interval(successes, trials, z=1.96):
    """Return a Wilson binomial interval."""
    fraction = successes / trials
    denominator = 1 + z**2 / trials
    centre = (fraction + z**2 / (2 * trials)) / denominator
    half_width = z * np.sqrt(fraction * (1 - fraction) / trials + z**2 / (4 * trials**2)) / denominator
    return centre - half_width, centre + half_width

validation = summary['three_model_validation']
amplitudes = ['0.3', '0.5', '1.0']
x = np.arange(len(amplitudes))
width = 0.36
fig, ax = plt.subplots(figsize=(8, 5))
for offset, window, label in [(-width / 2, 'continuous', 'continuous'), (width / 2, 'tess-like', 'TESS-like gaps')]:
    counts = np.array([validation[window]['true_positive_by_amplitude'][amplitude] for amplitude in amplitudes])
    rates = counts / campaign['positive_per_amplitude_per_window']
    intervals = np.array([wilson_interval(count, campaign['positive_per_amplitude_per_window']) for count in counts])
    errors = np.vstack((rates - intervals[:, 0], intervals[:, 1] - rates))
    ax.bar(x + offset, rates, width, yerr=errors, capsize=4, label=label)
ax.set_xticks(x, [f'{value}×' for value in amplitudes])
ax.set(xlabel='Injected oscillation amplitude relative to prediction', ylabel='True-positive rate', ylim=(0, 1.08), title='Recovery depends strongly on oscillation amplitude')
ax.legend();

The headline TPR is low largely because two thirds of the injected oscillators are intentionally suppressed:

- continuous: 35/72 = 48.6% overall, but 23/24 = 95.8% at full amplitude;
- TESS-like gaps: 28/72 = 38.9% overall, but 19/24 = 79.2% at full amplitude.

All 81 false negatives across both windows preferred the granulation model. Their median oscillation probability was about 0.045, so most were not cases sitting just below the 0.45 threshold.

In [ ]:
false_negative = validation['false_negatives_across_both_windows']
labels = ['noise', 'granulation', 'oscillation']
probabilities = [false_negative['median_probabilities'][label] for label in labels]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, probabilities, color=['0.4', 'tab:orange', 'tab:blue'])
ax.set(ylabel='Median posterior probability', ylim=(0, 1.05), title=f"Posterior composition of {false_negative['count']} false negatives");

## 4. Why not remove the granulation-only model?

The proposed two-model comparison keeps pure noise and the full noise+granulation+oscillation model. A granulation-only PSD then has no correct model available. The full model wins because it is the only remaining model containing granulation, even if its oscillation envelope is not supported.

Recombining the same evidence estimates gives an apparent 100% TPR, but also a 50% false-positive rate: every one of the 24 granulation-only validation spectra is called a detection. No tested threshold satisfies the 5% false-positive constraint.

In [ ]:
three = validation['tess-like']
two = summary['noise_oscillation_validation']['tess-like']
model_sets = ['Three models', 'Noise vs\noscillation']
x = np.arange(2)
width = 0.36
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(x - width / 2, [three['true_positive_rate'], two['true_positive_rate']], width, label='true-positive rate')
ax.bar(x + width / 2, [three['false_positive_rate'], two['false_positive_rate']], width, label='false-positive rate')
ax.axhline(0.05, color='black', ls='--', lw=1, label='5% FPR limit')
ax.set_xticks(x, model_sets)
ax.set(ylabel='Fraction', ylim=(0, 1.05), title='Removing granulation detects the background')
ax.legend();

The correct binary interpretation keeps both background models in the null:

$$
P(\mathrm{no\ detection}\mid D)
=P(\mathrm{noise}\mid D)+P(\mathrm{granulation}\mid D).
$$

Here “granulation” means that no visible oscillation envelope is required by the measured PSD. It does not mean that the star is physically incapable of oscillating.

## 5. Following up the observing-window result

Across all matched validation pairs in the first campaign, gaps caused 13 detections to be lost and 6 to be gained. The exact paired sign test below was not decisive on its own. A larger full-amplitude study then localized the loss to noisy low-luminosity red giants observed with a periodic 2.5-day momentum-dump pattern.

The periodic gaps redistribute power through the spectral window. Their alias spacing, $4.63\,\mu\mathrm{Hz}$, is close to half the simulated red giant's $\Delta\nu$. Passing every predicted model through the same target-specific window recovered the difficult population, as shown in the second plot.

In [ ]:
paired = validation['paired_probability_change']
discordant = paired['detections_lost'] + paired['detections_gained']
p_value = binomtest(min(paired['detections_lost'], paired['detections_gained']), discordant, 0.5).pvalue
print(f"Initial campaign: {paired['detections_lost']} lost, {paired['detections_gained']} gained; paired p={p_value:.3f}")

replay = json.loads((data_dir / 'window_aware_rgb_replay.json').read_text())
profile_names = ['continuous-current', 'gapped-current', 'gapped-window-aware']
labels = ['Continuous\ncurrent model', 'Gapped\ncurrent model', 'Gapped\nwindow-aware model']
rates = [replay['profiles'][name]['true_positive_rate'] for name in profile_names]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(labels, rates, color=['0.5', 'tab:orange', 'tab:blue'])
ax.set(ylabel='True-positive rate', ylim=(0, 1.05), title='Window-aware predictions recover the difficult RGB population')
for index, value in enumerate(rates):
    ax.text(index, value + 0.02, f'{value:.1%}', ha='center')

## Conclusions

1. The low overall TPR is mainly driven by deliberately suppressed $0.3\times$ and $0.5\times$ signals.
2. Full-amplitude continuous recovery is high.
3. Removing the granulation model creates detections of granulation and violates the false-positive constraint.
4. A larger paired study found a specific spectral-window failure for noisy red giants, rather than a generic duty-cycle loss.
5. Passing the predicted spectra through the target's measured window recovered 31/32 difficult cases, matching the continuous result, without increasing false positives in the paired control study.

The three-model detector is therefore retained, and real TESS validation uses the target-specific window-aware forward model. The likelihood still treats the binned Fourier powers as independent Gamma variables, so real-data probability calibration remains the next empirical check.